웹 크롤링(Web Crawling)이란?
- 프로그램이 웹 브라우저를 대신하여 인터넷에 있는 웹 페이지에 자동으로 접속하고<br>
    그 안에 있는 데이터를 수집하는 과정을 말합니다.

- 특징
    1. 사람이 직접 클릭하지 않아도 자동으로 동작합니다.
    2. 여러 웹 페이지를 반복적으로 수집할 수 있습니다.
    3. 웹 서버와 HTTP/HTTPS 통신을 사용합니다.
    4. 주로 HTML, JSON, XML 형식의 데이터를 대상으로 합니다.
---
<br>

웹 스크래핑(Scraping)이란?
- 이미 가져온 웹 페이지에서 필요한 데이터만 골라서 추출하는 과정입니다.
---
<br>

- 웹 크롤링 영역 (수집 및 연결)
    1. 요청(Request): 특정 주소(URL)에 접속을 시도합니다.
    2. 응답(Response): 서버로부터 페이지 전체 소스(HTML)를 받아옵니다.
- 웹 스크래핑 영역 (분석 및 추출)
    3. 파싱(Parsing): 복잡한 HTML 코드를 컴퓨터가 이해하기 쉽게 구조적으로 분석합니다.
    4. 데이터 추출(Extraction): 분석된 구조 속에서 '뉴스 제목', '가격' 등 특정 데이터만 쏙 뽑아냅니다.
---
<br>

- 필요 라이브러리

    requests # 웹 서버에 HTTP/HTTPS 요청을 보내고 응답을 받기 위한 라이브러리입니다.
    BeautifulSoup # 서버로부터 받은 HTML 문서를 파싱하여, 원하는 데이터(태그·속성 등)를 추출하기 위한 라이브러리입니다.

In [1]:
%pip install requests beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
from bs4 import BeautifulSoup
import urllib3

# SSL 검증 관련 경고 무시
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

url = "https://raw.githubusercontent.com/hiksyksy1234/json/main/index.html"

response = requests.get(url, verify=False) # verify는 SSL 인증 검사 옵션, 보안상 주의 필요

## SSL(Secure Sockets Layer) 인증서 검사란
- SSL 인증서 검사는 웹 브라우저나 Python의 `requests` 라이브러리가 서버에 접속할 때,  
서버가 제시한 인증서가 **신뢰할 수 있는 인증기관(CA)** 에 의해 발급되었고  
**접속한 도메인과 일치하는지** 등을 확인하여  
통신대상 서버가 **사칭이 아닌 '진짜 서버'인지 검증하는 디지털 신분증 확인 절차**입니다.

- 이 인증서 검사가 **성공한 이후**, TLS 핸드셰이크가 완료되며  
암호화된 **HTTPS 통신이 시작됩니다.**
---
## 검사의 3대 핵심 목적
서버와 통신을 시작하기 전, 다음 세가지 보안 요소를 확인하거나 보장합니다.

1. **기밀성 (Confidentiality)**  
전송되는 데이터가 암호화되어 제3자가 데이터를 가로채더라도  
내용을 읽을 수 없도록 보장합니다.  
→ 인증서 검증 성공 후, TLS 암호화 통신을 통해 달성됩니다.

2. **무결성 (Integrity)**  
데이터가 전송 도중에 위조되거나 변조되지 않았는지를 검증합니다.  
→ 메시지 인증 코드(MAC) 및 암호화 알고리즘을 통해 보장됩니다.

3. **인증 (Authentication)**  
접속한 웹사이트가 사칭 사이트가 아닌,  
실제 해당 도메인의 **진짜 서버인지 확인**합니다.  
→ SSL 인증서 검사의 **직접적인 핵심 목적**입니다.

> 정리하면,  
> **인증(Authentication)** 은 인증서 검사의 목적이고,  
> **기밀성·무결성** 은 인증서 검증 이후 이루어진느 HTTPS 통신의 보안 속성입니다.
---
## Python에서 `verify` 옵션의 의미
- `requests.get()` 함수의 `verify` 인자는  
**SSL 인증서 검사를 수행할지 여부를 결정하는 옵션**입니다.

| **구분** | **`verify=True` (권장)** | **`verify=False` (주의)** |
| :--- | :--- | :--- |
| **비유** | 신분증 검사 후 입장 | 신분증 확인 없이 프리패스 |
| **의미** | 서버 인증서를 검증한 후 통신 | 인증서 검증 없이 통신 |
| **장점** | 보안 사고 예방<br>데이터 안정성 보장 | 인증서 오류로 인한 코드 중단 방지 |
| **단점** | 인증서 설정 오류 시 접속 불가 | 중간자 공격(MITM)에 취약 |
| **적용** | 실제 서비스<br>결제 시스템<br>개인정보 전송 | 로컬 테스트<br>학습용 크롤링<br>단순 데이터 수집 |
---
## 중요한 주의 사항
- `verify=False`를 사용하더라도 **통신 자체는 TLS로 암호화**됩니다.
- 그러나 **누구와 암호화 통신을 하고 있는지는 검증하지 않기 때문에,**  
    공격자가 중간에서 통신을 가로채는 **중간자 공격** 에 노출될 수 있습니다.
- 따라서 `verify=False`는 **학습·테스트 목적에서만 제한적으로 사용**해야 합니다.
---
TLS (Transport Layer Seacurity)란?

TLS는 인터넷 통신을 안전하게 보호하기 위한 암호화 프로토콜입니다.

"웹이서데이터를 암호화해서 주고받는 약속(규칙)" 입니다.

현재 우리가 사용하는 HTTPS의 핵심 기술입니다.

In [3]:
print(response.status_code)

# 200(OK)
# 403(Forbidden): 권한 없음
# 404(Not Found): 찾을 수 없음(주소 잘못되거나)
# 500(Internal Server Error): 서버 오류 (서버측에 문제 있음)

200


In [4]:
print(response.text)

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>나의 크롤링 연습 사이트</title>
</head>
<body>
    <h1>도서 베스트셀러 목록</h1>
    <ul id="book-list">
        <li class="book-item">
            <span class="title">파이썬 데이터 분석 기초</span>
            <span class="price">25,000원</span>
        </li>
        <li class="book-item">
            <span class="title">머신러닝 완벽 가이드</span>
            <span class="price">38,000원</span>
        </li>
        <li class="book-item">
            <span class="title">웹 크롤링 입문</span>
            <span class="price">22,000원</span>
        </li>
    </ul>

    <h1>사용자 후기</h1>
    <div class="review-section">
        <p class="review">강의 내용이 너무 알차서 좋았습니다!</p>
        <p class="review">데이터 분석 입문자에게 강력 추천합니다.</p>
    </div>
</body>
</html>



In [5]:
if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')

    # find()로 select, get_text()로 텍스트노드 추출
    title = soup.find('h1').get_text()
    print(f"추출된 제목: {title}")
    
    book_titles = [t.get_text() for t in soup.select('.title')]
    print(f"도서 목록 {book_titles}")
else:
    print(f"페이지를 불러올 수 없습니다. 상태 코드: {response.status_code}")

추출된 제목: 도서 베스트셀러 목록
도서 목록 ['파이썬 데이터 분석 기초', '머신러닝 완벽 가이드', '웹 크롤링 입문']


In [6]:
# df.to_excel()에 필요한 엔진 라이브러리
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd

if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # 데이터를 담을 빈 리스트 생성
    book_data = []
    
    # soup : HTML 문서 전체
    # soup.select() : HTML 문서 전체에서 클래스 이름이 book-item인 모든 요소를 찾아 리스트 형태로 가져오라는 명령어입니다.
    items = soup.select('.book-item')
    
    for item in items:
        # 도서명과 가격 추출
        name_el = item.select_one('.title') # select_one() : 조건에 맞는 것 중 가장 먼저 나오는 하나만 가져옵니다
        price_el = item.select_one('.price')
        if name_el and price_el:
            name = name_el.get_text()
            price_text = price_el.get_text()
        
        # 분석을 위해 가격에서 '원'과 ','를 제거하고 정수(int)로 변환
        price = int(price_text.replace('원', '').replace(',', ''))
        
        # 딕셔너리 형태로 리스트에 추가
        book_data.append({
            '도서명': name,
            '가격': price
        })
    
    # 2. 리스트를 DataFrame으로 변환
    df = pd.DataFrame(book_data)
    
    # 결과 출력
    print("--- 생성된 DataFrame ---")
    print(df)
    
    # 엑셀 파일로 저장
    df.to_excel("books.xlsx", index=False)
else:
    print(f"페이지를 불러올 수 없습니다. 상태 코드: {response.status_code}")

--- 생성된 DataFrame ---
             도서명     가격
0  파이썬 데이터 분석 기초  25000
1    머신러닝 완벽 가이드  38000
2       웹 크롤링 입문  22000


---
코드 없이 엑셀 내용을 보려면 확장 프로그램 'Excel Viewer'를 설치하세요

In [8]:
import pandas as pd

file_path = 'books.xlsx'
df = pd.read_excel(file_path)

print("--- 불러온 데이터 확인 ---")
print(df.head())
print("-" * 30)

--- 불러온 데이터 확인 ---
             도서명     가격
0  파이썬 데이터 분석 기초  25000
1    머신러닝 완벽 가이드  38000
2       웹 크롤링 입문  22000
------------------------------


In [9]:
print(f"전체 도서 평균 가격: {df['가격'].mean():,.0f}원")
result = df[df['가격'] >= 30000]
print(f"--- 3만원 이상 도서 (총 {len(result)}권) ---")
if not result.empty:
    print(result)
else:
    print("3만원 이상의 도서가 없음")

전체 도서 평균 가격: 28,333원
--- 3만원 이상 도서 (총 1권) ---
           도서명     가격
1  머신러닝 완벽 가이드  38000


---
h1 태그들의 내용 추출하기

In [ ]:
soup = BeautifulSoup(response.text, 'html.parser')

h1_tags = soup.select('h1')
h1_contents = [tag.get_text() for tag in h1_tags]
print(f"추출된 h1 내용: {h1_contents}")

추출된 h1 내용: ['도서 베스트셀러 목록', '사용자 후기']


---
사용자 후기 추출하기

In [23]:
review_elements = soup.select('p.review')

user_reviews = [reply.get_text() for reply in review_elements]

print(f"--- 추출된 사용자 후기 ({len(user_reviews)}건) ---")
for i, review in enumerate(user_reviews, 1):
    print(f"{i}. {review}")

--- 추출된 사용자 후기 (2건) ---
1. 강의 내용이 너무 알차서 좋았습니다!
2. 데이터 분석 입문자에게 강력 추천합니다.


---
추출된 사용자 후기를 데이터 프레임으로 만들고 엑셀로 저장하기

In [24]:
df_reviews = pd.DataFrame(user_reviews, columns=['사용자_후기'])

print('\n--- 데이터프레임 형태 ---')
print(df_reviews)

df_reviews.to_excel('user_reviews.xlsx', index=False)


--- 데이터프레임 형태 ---
                   사용자_후기
0    강의 내용이 너무 알차서 좋았습니다!
1  데이터 분석 입문자에게 강력 추천합니다.
